The first audio model (v1) overfitted: it learned the training data almost
perfectly but reached only 58.8% on the validation set. In this notebook, we
train the same model again with data augmentation and regularisation
(SpecAugment, random time shift, mixup, label smoothing and dropout).

The data, the speaker split and the network are the same as in notebook 01.
The new model is kept only if it is better than v1 on the validation set.

**Results:** best epoch 33, validation accuracy 66.8%, test accuracy 67.8%.
This is the audio model used in the report.

The notebook also saves the predictions and the training history, which are
used in all analyses. The outputs are from the original run.

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import json, random, shutil, time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import accuracy_score, f1_score

BASE = Path("/content/drive/MyDrive/Affective Project/affective computing")
FEATURES = BASE / "audio_features_v1"
LOCAL = Path("/content/audio_features_v1"); LOCAL.mkdir(exist_ok=True)
for split in ["train", "validation", "test"]:
    if not (LOCAL / f"{split}.npz").exists():
        shutil.copy2(FEATURES / f"{split}.npz", LOCAL / f"{split}.npz")

RUN_DIR = BASE / "audio_runs" / ("v2_augment_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
V1_VAL_F1 = 0.5886            # validation macro-F1 of the current audio model

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| run folder:", RUN_DIR)

Device: cuda | run folder: /content/drive/MyDrive/Affective Project/affective computing/audio_runs/v2_augment_20261026_190959


In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

def spec_augment(x, n_freq=2, max_f=24, n_time=2, max_t=40):
    """Mask random frequency bands (rows) and time segments (columns). x: [H, W] in [0, 1]."""
    H, W = x.shape
    x = x.clone()
    for _ in range(n_freq):
        f = random.randint(0, max_f); f0 = random.randint(0, H - f)
        x[f0:f0 + f, :] = 0.0
    for _ in range(n_time):
        t = random.randint(0, max_t); t0 = random.randint(0, W - t)
        x[:, t0:t0 + t] = 0.0
    return x

class SpecDataset(Dataset):
    def __init__(self, split, augment=False):
        with np.load(LOCAL / f"{split}.npz", allow_pickle=False) as d:
            self.images, self.labels = d["images"], d["labels"]
            self.filenames, self.speakers = d["filenames"], d["speakers"]
            self.config = json.loads(d["config"].item())
        self.augment = augment
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        x = torch.from_numpy(self.images[i].copy()).float() / 255.0
        if self.augment:
            x = torch.roll(x, shifts=random.randint(-20, 20), dims=1)   # random time shift
            x = spec_augment(x)
        x = (x.unsqueeze(0).repeat(3, 1, 1) - MEAN) / STD
        return x, int(self.labels[i])

train_ds = SpecDataset("train", augment=True)
val_ds = SpecDataset("validation")
test_ds = SpecDataset("test")
classes = train_ds.config["classes"]
assert set(train_ds.speakers).isdisjoint(val_ds.speakers) and set(val_ds.speakers).isdisjoint(test_ds.speakers)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
print("Train / val / test:", len(train_ds), len(val_ds), len(test_ds))

Train / val / test: 4667 1222 1552


In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(classes)))
model = model.to(device)

optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if not n.startswith("fc.")], "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-3)
EPOCHS, PATIENCE = 40, 8
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
LABEL_SMOOTHING, MIXUP_ALPHA = 0.1, 0.2

def soft_ce(logits, target_probs):
    return -(target_probs * F.log_softmax(logits, dim=1)).sum(1).mean()

def train_one_epoch():
    model.train(); total = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        t = F.one_hot(y, len(classes)).float() * (1 - LABEL_SMOOTHING) + LABEL_SMOOTHING / len(classes)
        lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
        idx = torch.randperm(x.size(0), device=device)
        x = lam * x + (1 - lam) * x[idx]
        t = lam * t + (1 - lam) * t[idx]
        optimizer.zero_grad(set_to_none=True)
        loss = soft_ce(model(x), t)
        loss.backward(); optimizer.step()
        total += loss.item() * y.size(0)
    return total / len(train_loader.dataset)

@torch.no_grad()
def evaluate(loader):
    model.eval(); probs, ys, loss = [], [], 0.0
    for x, y in loader:
        logits = model(x.to(device))
        loss += F.cross_entropy(logits, y.to(device), reduction="sum").item()
        probs.append(torch.softmax(logits, 1).cpu().numpy()); ys.append(y.numpy())
    P, Y = np.concatenate(probs), np.concatenate(ys)
    pred = P.argmax(1)
    return dict(loss=loss / len(Y), acc=accuracy_score(Y, pred),
                f1=f1_score(Y, pred, average="macro"), P=P, Y=Y)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 152MB/s]


In [ ]:
history, best_f1, best_epoch, wait = [], -1.0, 0, 0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_one_epoch()
    tr = evaluate(DataLoader(SpecDataset("train"), batch_size=64))
    va = evaluate(val_loader)
    scheduler.step()
    history.append(dict(epoch=epoch, train_loss=tr["loss"], train_accuracy=tr["acc"], train_macro_f1=tr["f1"],
                        val_loss=va["loss"], val_accuracy=va["acc"], val_macro_f1=va["f1"]))
    pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
    if va["f1"] > best_f1:
        best_f1, best_epoch, wait = va["f1"], epoch, 0
        torch.save({"model_state_dict": model.state_dict(), "classes": classes, "config": train_ds.config,
                    "epoch": epoch, "val_macro_f1": float(va["f1"])}, RUN_DIR / "best_resnet18.pt")
    else:
        wait += 1
    print(f"Epoch {epoch:02d} | train acc {tr['acc']:.3f} | val acc {va['acc']:.3f} "
          f"| val F1 {va['f1']:.3f} | val loss {va['loss']:.3f} | {time.time() - t0:.0f}s")
    if wait >= PATIENCE:
        print("Early stopping"); break

print(f"\nBest epoch {best_epoch}: validation macro-F1 {best_f1:.4f} (v1: {V1_VAL_F1:.4f})")

Epoch 01 | train acc 0.421 | val acc 0.386 | val F1 0.357 | val loss 1.481 | 30s
Epoch 02 | train acc 0.499 | val acc 0.470 | val F1 0.447 | val loss 1.359 | 27s
Epoch 03 | train acc 0.549 | val acc 0.510 | val F1 0.491 | val loss 1.261 | 27s
Epoch 04 | train acc 0.611 | val acc 0.561 | val F1 0.552 | val loss 1.172 | 27s
Epoch 05 | train acc 0.605 | val acc 0.559 | val F1 0.545 | val loss 1.245 | 27s
Epoch 06 | train acc 0.598 | val acc 0.520 | val F1 0.492 | val loss 1.217 | 28s
Epoch 07 | train acc 0.687 | val acc 0.601 | val F1 0.590 | val loss 1.086 | 28s
Epoch 08 | train acc 0.678 | val acc 0.577 | val F1 0.564 | val loss 1.123 | 34s
Epoch 09 | train acc 0.657 | val acc 0.574 | val F1 0.536 | val loss 1.206 | 28s
Epoch 10 | train acc 0.755 | val acc 0.610 | val F1 0.609 | val loss 1.051 | 28s
Epoch 11 | train acc 0.731 | val acc 0.588 | val F1 0.564 | val loss 1.119 | 28s
Epoch 12 | train acc 0.769 | val acc 0.574 | val F1 0.582 | val loss 1.096 | 28s
Epoch 13 | train acc 0.791 |

In [ ]:
# Decision based ONLY on validation
ACCEPT = best_f1 > V1_VAL_F1
print("v2 is better on validation -> use v2" if ACCEPT else "v2 is NOT better on validation -> keep v1")

v2 is better on validation -> use v2


In [ ]:
assert ACCEPT, "v2 was not better on validation: keep the v1 model and do not export."
ck = torch.load(RUN_DIR / "best_resnet18.pt", map_location=device, weights_only=True)
model.load_state_dict(ck["model_state_dict"])

def to_df(ds, res):
    pred = res["P"].argmax(1)
    df = pd.DataFrame({"filename": ds.filenames, "Speaker": ds.speakers,
                       "true_label": [classes[i] for i in res["Y"]],
                       "predicted_label": [classes[i] for i in pred],
                       "correct": res["Y"] == pred})
    for i, c in enumerate(classes):
        df[f"prob_{c}"] = res["P"][:, i]
    return df

val_res, test_res = evaluate(val_loader), evaluate(test_loader)
to_df(val_ds, val_res).to_csv(RUN_DIR / "audio_v2_validation_predictions.csv", index=False)
to_df(test_ds, test_res).to_csv(RUN_DIR / "audio_v2_test_predictions.csv", index=False)
print(f"Validation: acc {val_res['acc']:.4f}, macro-F1 {val_res['f1']:.4f}")
print(f"Test:       acc {test_res['acc']:.4f}, macro-F1 {test_res['f1']:.4f}")
for name in ["audio_v2_validation_predictions.csv", "audio_v2_test_predictions.csv", "history.csv"]:
    files.download(str(RUN_DIR / name))

Validation: acc 0.6678, macro-F1 0.6663
Test:       acc 0.6778, macro-F1 0.6768


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>